# UrbanVendor-Vision v2 · entrenamiento sin fugas y a escala nativa

<a href="https://colab.research.google.com/github/Kzamudioq/urbanvendor-vision/blob/main/notebooks/entrenar_v2_colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"></a>

**Runtime → Change runtime type → T4 GPU.** En Google Drive, la carpeta `UrbanVendor-Vision/etiquetas` debe tener las fotos `DJI_xxxx.JPG` y sus `DJI_xxxx.json` de Labelme, y `UrbanVendor-Vision/ortofo` la ortofoto.

Qué cambia frente a v1:

| | v1 | v2 |
|---|---|---|
| Escala de entrenamiento | foto reducida a 0,4× y luego a imgsz 1500 → 10,7 cm/píxel | teselas de 1024 px sin reducir → 3,6 cm/píxel |
| Validación | 2 fotos, una a 21 m y otra a 26 m de fotos de entrenamiento | 2 zonas del vuelo a más de 140 m: se entrena en una y se evalúa en la otra |
| Aumentos | 3 copias por foto en Roboflow | en cada época: rotación libre (vista nadir), volteos, color |
| Ortofoto | teselas de ~1024 px a imgsz 1500 → objetos 4,5× más grandes que en entrenamiento | ventanas de 1024 px a imgsz 1024 → misma escala |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/UrbanVendor-Vision'
ETIQUETAS = f'{BASE}/etiquetas'
ORTO = f'{BASE}/ortofo/la_mariposa.tif'
SALIDA = f'{BASE}/v2'
!git clone -q https://github.com/Kzamudioq/urbanvendor-vision.git
%cd urbanvendor-vision
!pip install -q -e ".[modelo]"

## 1. Escala de las fotos
El GSD sale de la altura y la cámara que guarda el dron en cada foto.

In [ ]:
import glob
foto = sorted(glob.glob(f'{ETIQUETAS}/*.JPG'))[0]
!python -m urbanvendor escala --foto "$foto" --gsd-raster 0.0345453 --imgsz 1024

## 2. Dataset en teselas nativas, agrupado por zona

In [ ]:
!python -m urbanvendor dataset --fotos "$ETIQUETAS" --salida /content/ds --lado 1024 --zona-m 140
!ls /content/ds

## 3. Validación cruzada por zona
Un modelo por pliegue: entrena con todas las zonas menos una y se evalúa en la que no vio. Esa es la cifra que se reporta.
`degrees=180` porque en vista nadir una sombrilla no tiene "arriba".

In [ ]:
import glob, os, pandas as pd
from ultralytics import YOLO
HP = dict(imgsz=1024, epochs=150, patience=40, batch=8, degrees=180, flipud=0.5, fliplr=0.5,
          scale=0.25, mosaic=1.0, close_mosaic=15, cos_lr=True, seed=0, plots=True)
filas = []
for cfg in sorted(glob.glob('/content/ds/pliegue_*.yaml')):
    nombre = os.path.basename(cfg)[:-5]
    m = YOLO('yolo26s-seg.pt')
    m.train(data=cfg, project=f'{SALIDA}/cv', name=nombre, exist_ok=True, **HP)
    v = m.val(data=cfg, imgsz=1024, split='val', plots=False)
    filas.append({'pliegue': nombre, 'mAP50_mascara': v.seg.map50, 'mAP50_95_mascara': v.seg.map,
                  'precision': v.seg.mp, 'recall': v.seg.mr})
cv = pd.DataFrame(filas); cv.to_csv(f'{SALIDA}/validacion_cruzada.csv', index=False); cv

## 4. Modelo final con todas las fotos
Las métricas que se publican son las de la validación cruzada; este modelo solo se usa para producir el mapa.

In [ ]:
!mkdir -p /content/ds/todo
todo = '/content/ds/todo.yaml'
zonas = sorted(glob.glob('/content/ds/zona_*/images'))
open(todo, 'w').write('train:\n' + ''.join(f'  - {z}\n' for z in zonas) + f'val: {zonas[0]}\nnc: 1\nnames: [puesto_sombrilla]\n')
final = YOLO('yolo26s-seg.pt')
final.train(data=todo, project=SALIDA, name='final', exist_ok=True, **{**HP, 'patience': 0, 'epochs': 120})
PESOS = f'{SALIDA}/final/weights/last.pt'

## 5. Ortofoto completa y densidad
Ventanas de 1024 px a imgsz 1024: el modelo ve los objetos al mismo tamaño que en entrenamiento.

In [ ]:
!python -m urbanvendor ortofoto --raster "$ORTO" --pesos "$PESOS" --imgsz 1024 --gsd-modelo 0.0356 --salida "$SALIDA/puestos_v2.geojson"
!python -m urbanvendor densidad --puestos "$SALIDA/puestos_v2.geojson" --lado 15 --salida "$SALIDA/densidad_v2.geojson"